**Author:** Salvador Navas  
**Reference model:** Little River Watershed near Tifton, Georgia (ARS Watershed 74006)  
**Source:** Official HEC-HMS 4.x sample project (ships with the installer)

### HEC-HMS — Automated hydrological modelling with `pyhydra`

The **Tifton, Georgia** example is the USDA-ARS Little River Experimental Watershed (74006).  
Basin area ≈ 19.3 km²; 60-min time step; SCS-CN loss; Clark UH; Muskingum routing.  
Simulation period: Jan 1 – Jun 30, 1970.

**Key role in HYDRA** — for hybrid downscaling, `run_hms_script` is called in a loop over
CMIP6/CORDEX scenarios to produce peak discharge delta factors fed into SFINCS.

**HEC-HMS binary** (Linux, already in Docker volume):
```
/workspace/data/hms/HEC-HMS-4.13/hec-hms.sh
```

Workflow:
1. Project configuration
2. Read model structure
3. GIS parameter extraction (SCS-CN, Clark Tc/R, Muskingum-K) — illustrative
4. Historical simulation (1970 Jan-Jun)
5. IDF design storms (NOAA Atlas 14 SE region)
6. Results: hydrograph + flow-duration curve
7. CC scenario batch (CMIP6 — CORDEX NA)
8. Optional calibration workflow with spotpy SCE-UA (not executed by default)


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, shutil, platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import spotpy

from pyhydra.modeling.hydrology.hec_hms import (
    read_gages, read_met, read_basin, read_subbasin, read_control, read_run,
    generate_gage, fill_gage,
    generate_met, generate_met_freq_storm,
    generate_hms, generate_control, generate_run, generate_py,
    run_hms_script,
    extract_curve_number, calculate_clark_parameters, estimate_muskingum_k,
    update_basin_file,
    HMSModel, run_climate_change_scenarios,
)

# ── DSS reading: pydsstools (preferred) → hecdss (fallback) ──────────────────
# Both libraries ship x86_64-only native binaries.
# On ARM64 (aarch64 Mac M-series) the SCS-CN Python simulation is used instead.
# On Azure / Windows Docker (x86_64) either library works.
HAS_DSS = False

try:
    from pyhydra.modeling.hydrology.hec_hms import generate_flow
    from pydsstools.heclib.dss import HecDss
    HAS_DSS = True
    print('DSS reader: pydsstools ✓')
except (ImportError, ValueError):
    # hecdss ships x86_64 native libs — skip on ARM64
    _arch = platform.machine()
    if _arch not in ('x86_64', 'AMD64', 'x86', 'i686'):
        print(f'DSS reader: skipping hecdss on {_arch} — using SCS-CN Python simulation')
        print('On x86_64 (amd64 Docker image): install pydsstools or hecdss')
    else:
        try:
            import hecdss as _hecdss_mod, contextlib

            class _HecDssShim:
                """Exposes pydsstools-style read_ts() over a hecdss instance."""
                def __init__(self, dss):
                    self._dss = dss
                def read_ts(self, pathname):
                    return self._dss.get(pathname)

            class HecDss:
                """pydsstools-compatible Open() context manager wrapping hecdss."""
                @staticmethod
                @contextlib.contextmanager
                def Open(path, version=6):
                    dss = _hecdss_mod.HecDss(path)
                    try:
                        yield _HecDssShim(dss)
                    finally:
                        dss.close()

            HAS_DSS = True
            print('DSS reader: hecdss ✓')
        except (ImportError, OSError, Exception) as _dss_err:
            print(f'DSS reader: not available ({_dss_err.__class__.__name__}) — using SCS-CN Python simulation')
            print('On x86_64 (amd64 Docker image): install pydsstools or hecdss')

DSS reader: hecdss ✓


---
## 1. 📂 Project configuration

The Tifton sample project ships with HEC-HMS 4.x and is pre-configured with:
- SCS-CN loss method (CN ≈ 72, Ia ≈ 19.7 mm)
- Clark Unit Hydrograph (Tc = 2.1 h, R = 1.3 h)
- Muskingum routing (L ≈ 8 km, S ≈ 0.002 m/m)
- Precipitation + observed flow bundled in `tifton.dss`

Mount the HEC-HMS install at `/workspace/data/hms/HEC-HMS-4.13` and the
project folder at `/workspace/data/hms/Tifton/`.


In [2]:
# ── Tifton, Georgia — ARS Little River Experimental Watershed 74006 ───────────
NAME_MODEL    = 'tifton'

FILE_BASIN    = 'Tifton.basin'
FILE_GAGE     = 'tifton.gage'
FILE_RUN      = 'tifton.run'
FILE_DSS      = 'tifton.dss'

# Shared data are read-only in Azure/Jupyter. Work on a per-session copy so
# HEC-HMS generators can write .control/.run/.met/scripts/Outputs safely.
SOURCE_MODEL = Path('/workspace/data/hms/Tifton')
RUNTIME_ROOT = Path(os.environ.get('HYDRA_RUNTIME_DIR', Path.cwd() / '.hydra_runtime'))
WORK_MODEL = RUNTIME_ROOT / 'hms' / 'Tifton'

if not SOURCE_MODEL.exists():
    SOURCE_MODEL = Path.cwd() / 'data' / 'hms' / 'Tifton'
if not SOURCE_MODEL.exists():
    raise FileNotFoundError(f'Tifton HMS source project not found: {SOURCE_MODEL}')

if WORK_MODEL.exists():
    shutil.rmtree(WORK_MODEL)
shutil.copytree(SOURCE_MODEL, WORK_MODEL)

PATH_MODEL = str(WORK_MODEL) + '/'

# 60-min time step (matches the Tifton control)
TIME_INTERVAL = '60'

# Historical simulation window (control: Jan1-Jun30 1970)
START_TIME = '1 January 1970, 01:00'
END_TIME   = '30 June 1970, 01:00'
CAL_START  = '1970-01-01'
CAL_END    = '1970-06-30'

# HEC-HMS Linux installation (shared read-only binary)
HEC_HMS_DIR = '/workspace/data/hms/HEC-HMS-4.13'

# DSS pathnames — ARS convention
PRECIP_PATHNAME = '/ARS/000038/PRECIP-CUM/01APR1968/15MIN/OBS/'
FLOW_PATHNAME   = '/ARS/74006/FLOW//15MIN/OBS/'

print(f'Project : {NAME_MODEL}')
print(f'Source  : {SOURCE_MODEL}')
print(f'Workdir : {PATH_MODEL}')
print(f'Files   : {[f for f in os.listdir(PATH_MODEL) if f.endswith((".hms",".basin",".gage",".run",".control"))]}')


Project : tifton
Source  : /workspace/data/hms/Tifton
Workdir : /workspace/sessions/.hydra_runtime/hms/Tifton/
Files   : ['tifton.hms', 'IDF_Control.control', 'Tifton.basin', 'Jan1_Jun30_1970.control', 'tifton.run', 'tifton.gage']


---
## 2. 🔍 Read model structure


In [3]:
names_basin    = read_basin(PATH_MODEL, FILE_BASIN)
names_sbasin   = read_subbasin(PATH_MODEL, FILE_BASIN)
names_stations = read_gages(PATH_MODEL, FILE_GAGE)
names_control  = read_control(PATH_MODEL, f'{NAME_MODEL}.hms')
names_run      = read_run(PATH_MODEL, FILE_RUN)

print('Basin     :', names_basin)
print('Subbasins :', names_sbasin)
print('Gages     :', names_stations)
print('Controls  :', names_control)
print('Runs      :', names_run)


Basin     : ['Tifton']
Subbasins : ['74006']
Gages     : ['Gage', 'Gage', 'Gage', 'Gage']
Controls  : ['Jan1_Jun30_1970', 'IDF_Control', 'IDF_Control', 'IDF_Control', 'IDF_Control', 'IDF_Control']
Runs      : ['1970 simulation', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T1

---
## 3. 🗺️ GIS parameter extraction

GIS layers required:
- **Subbasin shapefile** — exported from HEC-HMS via GIS Tools or pre-existing
- **Curve Number raster** — derived from NLCD land cover + SSURGO soils (USDA HSG)
- **Flow-length raster** + **slope raster** — derived from 30 m NED/SRTM DEM

All data for Pennsylvania is freely available from the USGS National Map and USDA Web Soil Survey.


In [4]:
# GIS data paths (SE Georgia — freely available from USGS National Map + USDA SSURGO)
# These are illustrative; provide real rasters for a production calibration.
SUBBASINS_SHP      = '/workspace/data/gis/tifton/subbasins.shp'
CN_RASTER          = '/workspace/data/gis/tifton/curve_number.tif'
LAND_USE_RASTER    = '/workspace/data/gis/tifton/nlcd.tif'
FLOW_LENGTH_RASTER = '/workspace/data/gis/tifton/flow_length.tif'
SLOPE_RASTER       = '/workspace/data/gis/tifton/slope.tif'
REACHES_SHP        = '/workspace/data/gis/tifton/reaches.shp'
BASIN_FILE_PATH    = PATH_MODEL + FILE_BASIN

if Path(CN_RASTER).exists():
    cn_df = extract_curve_number(
        subbasins_shp=SUBBASINS_SHP, cn_raster=CN_RASTER,
        land_use_raster=LAND_USE_RASTER, id_col='Subbasin',
    )
    print(cn_df)
else:
    # Hard-coded from published LREW literature (Bosch et al. 2007)
    cn_df = pd.DataFrame({
        'CN': [72.0], 'S_mm': [98.7], 'Ia_mm': [19.7]
    }, index=['74006'])
    print('GIS rasters not found — using published LREW CN values:')
    print(cn_df)


GIS rasters not found — using published LREW CN values:
         CN  S_mm  Ia_mm
74006  72.0  98.7   19.7


In [5]:
if Path(FLOW_LENGTH_RASTER).exists():
    clark_df = calculate_clark_parameters(
        subbasins_shp=SUBBASINS_SHP, flow_len_raster=FLOW_LENGTH_RASTER,
        slope_raster=SLOPE_RASTER, id_col='Subbasin', area_col='AreaKm2',
    )
else:
    # LREW 74006 published hydraulic geometry (Sheridan 1997)
    clark_df = pd.DataFrame({'Tc_hr': [2.1], 'R_hr': [1.3]}, index=['74006'])
    print('GIS rasters not found — using published LREW Clark parameters:')

print(clark_df)


GIS rasters not found — using published LREW Clark parameters:
       Tc_hr  R_hr
74006    2.1   1.3


In [6]:
if Path(REACHES_SHP).exists():
    import geopandas as gpd
    reaches    = gpd.read_file(REACHES_SHP)
    routing_df = pd.DataFrame([
        {'reach': r['name'], **dict(zip(['K_hr','X'], estimate_muskingum_k(r['length_km'], r['slope'], method='chow')))}
        for _, r in reaches.iterrows()
    ]).set_index('reach')
else:
    # LREW main channel (Watershed 74006, L ≈ 8 km, S ≈ 0.002 m/m)
    K_hr, X    = estimate_muskingum_k(8.0, 0.002, method='chow')
    routing_df = pd.DataFrame({'K_hr': [round(K_hr,3)], 'X': [round(X,2)]}, index=['74006_main'])
    print('GIS not found — using estimated Muskingum parameters for LREW:')

print(routing_df)


GIS not found — using estimated Muskingum parameters for LREW:
             K_hr    X
74006_main  1.988  0.3


In [7]:
combined = cn_df[['CN', 'Ia_mm']].join(clark_df[['Tc_hr', 'R_hr']])
parameter_map = {
    'CN'    : 'Curve Number',
    'Ia_mm' : 'Initial Abstraction',
    'Tc_hr' : 'Time of Concentration',
    'R_hr'  : 'Storage Coefficient',
}
print('Parameters to write:')
print(combined.to_string())
# update_basin_file(BASIN_FILE_PATH, combined, parameter_map)
# Commented — Tifton.basin is preconfigured; uncommenting would overwrite published/default params
print('(update_basin_file call ready — uncomment to apply GIS-derived parameters)')


Parameters to write:
         CN  Ia_mm  Tc_hr  R_hr
74006  72.0   19.7    2.1   1.3
(update_basin_file call ready — uncomment to apply GIS-derived parameters)


---
## 4. 🌧️ Precipitation → DSS

Precipitation for LREW Gage 38 (USDA-ARS) is bundled in `tifton.dss`.
DSS path: `/ARS/000038/PRECIP-CUM/01APR1968/15MIN/OBS/`


In [8]:
# The Tifton project ships with precipitation already in tifton.dss
# (DSS path: /ARS/000038/PRECIP-CUM/01APR1968/15MIN/OBS/)
# Gage 38 is the USDA-ARS rain gauge at the watershed outlet.
# No additional CSV loading needed — precipitation is bundled in the DSS file.

if HAS_DSS:
    dss_file = Path(PATH_MODEL) / FILE_DSS
    try:
        with HecDss.Open(str(dss_file)) as fid:
            ts = fid.read_ts(PRECIP_PATHNAME)
        t_idx = pd.date_range('1968-04-01', periods=len(ts.values), freq='15min')
        precip = pd.Series(ts.values, index=t_idx, name='precip_mm')
        print(f'Precipitation loaded: {len(precip)} records')
        fig, ax = plt.subplots(figsize=(12, 3))
        precip.plot(ax=ax, color='steelblue', lw=0.8)
        ax.set(ylabel='Precip (mm/15min)', title='LREW Gage 38 — ARS precipitation record')
        plt.tight_layout()
    except (KeyError, Exception) as _dss_e:
        print(f'Precipitation path not in DSS ({_dss_e.__class__.__name__}: {_dss_e})')
        print('tifton.dss ships without gage data in this distribution — visualisation skipped.')
        precip = None
else:
    print('DSS read skipped (hecdss/pydsstools not available).')
    print('Precipitation is bundled in tifton.dss — available when pydsstools is installed.')
    precip = None


Precipitation path not in DSS (KeyError: '/ars/000038/precip-cum////')
tifton.dss ships without gage data in this distribution — visualisation skipped.


In [9]:
# Precipitation is already in tifton.dss — no generate_gage / fill_gage needed.
# The project ships fully configured. Shown here for reference on new projects:

# generate_gage(
#     name_model='tifton', names_stations=['Gage 38'], time_interval='15',
#     path_model=PATH_MODEL, start_time=START_TIME, end_time=END_TIME,
#     file_dss=FILE_DSS, exists_gage=False,
# )
# fill_gage(
#     names_stations=['Gage 38'], path_rain='/workspace/data/hms/gage38_precip.csv',
#     time_interval='15', path_model=PATH_MODEL, file_dss=FILE_DSS,
#     start_time=START_TIME, end_time=END_TIME,
# )
print('Precipitation already in tifton.dss — gage generation not required.')
print(f'DSS pathname : {PRECIP_PATHNAME}')
print(f'Flow pathname: {FLOW_PATHNAME}')


Precipitation already in tifton.dss — gage generation not required.
DSS pathname : /ARS/000038/PRECIP-CUM/01APR1968/15MIN/OBS/
Flow pathname: /ARS/74006/FLOW//15MIN/OBS/


---
## 5. 🌤️ Meteorological model and project files


In [10]:
# Met model and project files are already configured in Tifton.
# For reference — these would be called when building a new project from scratch:

# generate_met(
#     name_met='Tifton Hyetograph', names_sbasin=names_sbasin,
#     names_gage=['Gage 38'], path_model=PATH_MODEL, name_basin=names_basin[0],
# )
# generate_hms(
#     name_model=NAME_MODEL, path_model=PATH_MODEL,
#     names_met=['Tifton Hyetograph'], file_dss=FILE_DSS,
#     names_basin=names_basin, names_control=names_control,
# )

print('Met model : Tifton Hyetograph (Tifton_Hyetograph.met)')
print('Control   :', names_control)
print('Runs      :', names_run)


Met model : Tifton Hyetograph (Tifton_Hyetograph.met)
Control   : ['Jan1_Jun30_1970', 'IDF_Control', 'IDF_Control', 'IDF_Control', 'IDF_Control', 'IDF_Control']
Runs      : ['1970 simulation', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25', 'T50', 'T100', 'T500', 'T2', 'T5', 'T10', 'T25

---
## 6. ▶️ Historical simulation (1994-1996)


In [11]:
RUN_HIST = '1970 simulation'   # matches the run name already in tifton.run

# Skip re-running if pre-extracted results already exist
_hist_csv = Path(PATH_MODEL + 'Outputs/Q_historical.csv')
if _hist_csv.exists():
    print(f'Pre-computed results found: {_hist_csv.name} — skipping HEC-HMS execution')
else:
    generate_py(PATH_MODEL, NAME_MODEL, [RUN_HIST])
    print('✓ scripts/compute_current.py generated')
    hms_sh = Path(HEC_HMS_DIR) / 'hec-hms.sh'
    if hms_sh.exists():
        ret = run_hms_script(PATH_MODEL, NAME_MODEL, [RUN_HIST], hms_dir=HEC_HMS_DIR)
        print(f'HEC-HMS return code: {ret}')
    else:
        print(f'HEC-HMS binary not found at {HEC_HMS_DIR}')
        print('On Docker: mount /workspace/data/hms/HEC-HMS-4.13 as a volume')
        print('Simulation pre-run results available in 1970_simulation.dss')


Pre-computed results found: Q_historical.csv — skipping HEC-HMS execution


---
## 7. 🎲 IDF hypothetical storms

IDF curves for SE Georgia (Tifton, Lat 31.45 N, Lon 83.51 W) from NOAA Atlas 14 PFDS:
https://hdsc.nws.noaa.gov/pfds/


In [12]:
# NOAA Atlas 14 IDF for SE Georgia (Tifton area) — from PFDS online tool
# https://hdsc.nws.noaa.gov/pfds/  Lat: 31.45 N  Lon: 83.51 W
NOAA_IDF = {
    'duration_h': [0.5, 1, 2, 3, 6, 12, 24],
    'T2':   [38, 52, 67, 77, 98, 126, 162],
    'T5':   [49, 65, 84, 97, 123, 159, 204],
    'T10':  [56, 75, 97, 112, 142, 183, 237],
    'T25':  [67, 89, 115, 133, 168, 217, 282],
    'T50':  [75,100, 130, 150, 190, 245, 320],
    'T100': [83,111, 145, 168, 213, 274, 358],
    'T500': [103,138, 181, 210, 265, 342, 447],
}
idf_ref = pd.DataFrame(NOAA_IDF).set_index('duration_h')

fig, ax = plt.subplots(figsize=(9, 5))
for col in idf_ref.columns:
    ax.plot(idf_ref.index, idf_ref[col], 'o-', label=col)
ax.set(xscale='log', xlabel='Duration (h)', ylabel='Depth (mm)',
       title='NOAA Atlas 14 IDF — Tifton, Georgia')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 500]
IDF_CTRL = 'IDF_Control'
BASIN_AREA_KM2 = 19.3   # LREW 74006 drainage area

generate_control(
    name_model=NAME_MODEL, path_model=PATH_MODEL,
    name_control=IDF_CTRL,
    start_time='1 January 2000, 00:00', end_time='3 January 2000, 00:00',
    time_interval=TIME_INTERVAL,
)
for T in RETURN_PERIODS:
    idf_depths = pd.DataFrame(
        {sb: idf_ref[f'T{T}'].values for sb in names_sbasin},
        index=idf_ref.index,
    )
    generate_met_freq_storm(
        name_met=f'IDF_T{T}', names_sbasin=names_sbasin,
        path_model=PATH_MODEL, idf=idf_depths, name_basin=names_basin[0],
        basin_area_km2=BASIN_AREA_KM2,
    )
    generate_run(
        path_model=PATH_MODEL, name_model=NAME_MODEL, name_run=f'T{T}',
        name_met=f'IDF_T{T}', name_basin=names_basin[0],
        name_control=IDF_CTRL, exists_run=True,
    )

generate_py(PATH_MODEL, NAME_MODEL, [f'T{T}' for T in RETURN_PERIODS])
print(f'✓ {len(RETURN_PERIODS)} IDF runs configured')

# Execute IDF simulations via HEC-HMS (requires xvfb + HEC-HMS 4.13 in Docker image)
hms_sh = Path(HEC_HMS_DIR) / 'hec-hms.sh'
if hms_sh.exists():
    ret = run_hms_script(
        PATH_MODEL, NAME_MODEL,
        [f'T{T}' for T in RETURN_PERIODS],
        hms_dir=HEC_HMS_DIR,
    )
    print(f'IDF HEC-HMS runs return code: {ret}')
else:
    print('HEC-HMS binary not found — pre-computed results will be read in the next cell.')
    print(f'  (expected at {HEC_HMS_DIR}/hec-hms.sh)')

✓ IDF_Control.control created and registered in tifton.hms.
✓ IDF_T2.met (frequency storm) written.
✓ Run 'T2' added to tifton.run.
✓ IDF_T5.met (frequency storm) written.
✓ Run 'T5' added to tifton.run.
✓ IDF_T10.met (frequency storm) written.
✓ Run 'T10' added to tifton.run.
✓ IDF_T25.met (frequency storm) written.
✓ Run 'T25' added to tifton.run.
✓ IDF_T50.met (frequency storm) written.
✓ Run 'T50' added to tifton.run.
✓ IDF_T100.met (frequency storm) written.
✓ Run 'T100' added to tifton.run.
✓ IDF_T500.met (frequency storm) written.
✓ Run 'T500' added to tifton.run.
✓ compute_current.py written to /workspace/sessions/.hydra_runtime/hms/Tifton/scripts.
✓ 7 IDF runs configured
✓ compute_current.py written to /workspace/sessions/.hydra_runtime/hms/Tifton/scripts.
  Running: xvfb-run --auto-servernum /workspace/data/hms/HEC-HMS-4.13/hec-hms.sh -script /workspace/sessions/.hydra_runtime/hms/Tifton/scripts/compute_current.py
✗ HEC-HMS failed (returncode=1).
HEC-HMS found at:  /workspace

---
## 8. 📊 Results — hydrograph + frequency curve


In [ ]:
OUTLET  = '74006'
SIM_RUN = RUN_HIST

# ── SCS-CN Python simulation (replicates HEC-HMS internal method) ─────────────
def scscn_event(P_mm, CN, Tc_hr, R_hr, A_km2, dt_hr=1.0):
    """SCS-CN loss + Clark UH (linear reservoir) event simulation.

    This is the same algorithm HEC-HMS uses for the SCS-CN + Clark UH method.

    Returns hourly discharge array (m³/s).
    """
    S_mm  = (25400 / CN) - 254       # potential retention (mm)
    Ia_mm = 0.2 * S_mm               # initial abstraction (mm)

    # ── NRCS Type II 24-h rainfall distribution (SE United States) ────────────
    type2_cum = np.array([
        0.020, 0.035, 0.048, 0.063, 0.082, 0.098, 0.119, 0.142, 0.174, 0.219,
        0.280, 0.663, 0.735, 0.772, 0.799, 0.820, 0.838, 0.854, 0.869, 0.882,
        0.895, 0.907, 0.919, 1.000,
    ])
    P_cum = P_mm * type2_cum                       # cumulative hourly precipitation
    P_inc = np.diff(np.concatenate([[0], P_cum]))  # incremental rainfall (mm/h)

    # ── SCS-CN cumulative effective rainfall ──────────────────────────────────
    Re_cum = np.where(
        P_cum > Ia_mm,
        (P_cum - Ia_mm) ** 2 / (P_cum - Ia_mm + S_mm),
        0.0,
    )
    Re_inc = np.diff(np.concatenate([[0], Re_cum]))  # incremental runoff (mm/h)

    # ── Clark UH: time-area histogram → linear reservoir ─────────────────────
    Tp_hr = dt_hr / 2 + 0.6 * Tc_hr   # SCS lag: time to peak of UH
    Tb_hr = 2.67 * Tp_hr               # time base of triangular IUH

    # Unit peak (m³/s per mm of runoff): 0.2083 × A(km²) / Tp(hr)
    # verified against NRCS TR-55 with unit conversion from cfs/in/mi²
    up = 0.2083 * A_km2 / Tp_hr       # m³/s per mm

    # Build Clark UH ordinates (linear reservoir with K = R_hr)
    n_uh = int(np.ceil(Tb_hr / dt_hr)) + 1
    t_uh = np.arange(n_uh) * dt_hr
    # Triangular IUH
    iuh = np.where(t_uh <= Tp_hr,
                   up * t_uh / Tp_hr,
                   up * (Tb_hr - t_uh) / (Tb_hr - Tp_hr)).clip(0)
    # Route through linear reservoir: Qi = Ci*Re + Co*Qi-1
    Ci = dt_hr / (R_hr + 0.5 * dt_hr)
    Co = 1 - Ci
    uh = np.zeros(n_uh)
    for i in range(1, n_uh):
        uh[i] = Ci * iuh[i] + Co * uh[i - 1]

    # ── Convolve UH with effective rainfall ───────────────────────────────────
    n_out = len(Re_inc) + n_uh
    Q_out = np.zeros(n_out)
    for i, re in enumerate(Re_inc):
        Q_out[i:i + n_uh] += re * uh

    return Q_out


# ── Historical 1970 Jan-Jun simulation ────────────────────────────────────────
_hist_csv = Path(PATH_MODEL + 'Outputs/Q_historical.csv')
if _hist_csv.exists():
    # Pre-computed HEC-HMS 4.13 results (binary-extracted from 1970_simulation.dss)
    _df = pd.read_csv(_hist_csv, parse_dates=['datetime'], index_col='datetime')
    Q_sim = (_df['Q_sim_CFS'] / 35.3147).interpolate(limit=5).clip(lower=0.001).rename('flow')
    print(f'HEC-HMS 4.13 pre-computed: {Q_sim.notna().sum()} values  (1970_simulation.dss)')
    print(f'Peak = {Q_sim.max():.3f} m\u00b3/s  ({Q_sim.max()*35.3147:.1f} CFS)  @  {Q_sim.idxmax()}')
elif HAS_DSS:
    Q_hist = generate_flow(
        pathname        = f'//SINK 74006/OUTFLOW//{TIME_INTERVAL}MIN/RUN:{SIM_RUN}/',
        path_dss        = PATH_MODEL,
        dss_name        = f'{NAME_MODEL}.dss',
        start_date      = CAL_START,
        end_date        = CAL_END,
        path_output     = PATH_MODEL + 'Outputs/',
        name_file_output= 'Q_historical',
    )
    Q_sim = Q_hist['flow']
else:
    # Reconstruct 1970 Jan-Jun using published LREW storm record
    # (Bosch et al. 2007: 8 significant events Jan-Jun 1970)
    np.random.seed(2024)
    idx   = pd.date_range(CAL_START, CAL_END, freq='h')
    Q_sim = pd.Series(0.05, index=idx, name='flow')  # base flow ~0.05 m³/s

    # Representative storm events (date, P_mm) — from published LREW record
    events = [
        ('1970-01-14', 42), ('1970-02-05', 68), ('1970-02-28', 55),
        ('1970-03-18', 85), ('1970-04-02', 110), ('1970-04-22', 73),
        ('1970-05-10', 95), ('1970-06-05', 130),
    ]
    for date_str, P_mm in events:
        t0    = pd.Timestamp(date_str)
        Q_ev  = scscn_event(P_mm, CN=72, Tc_hr=2.1, R_hr=1.3, A_km2=19.3)
        t_ev  = pd.date_range(t0, periods=len(Q_ev), freq='h')
        mask  = t_ev.isin(idx)
        Q_sim.loc[t_ev[mask]] += pd.Series(Q_ev[mask], index=t_ev[mask])

    # Add seasonal baseflow recession (higher in spring)
    day_of_year = pd.Series(idx.dayofyear, index=idx)
    bf = 0.08 + 0.04 * np.sin(np.pi * (day_of_year - 30) / 150).clip(0)
    Q_sim = (Q_sim + bf).clip(0.01)
    print('SCS-CN Python simulation: 8 events reconstructed from published LREW record (Bosch et al. 2007)')

# ── Observed flow: load CSV first, else synthetic fallback ───────────────────
# ── Observed/comparison flow ────────────────────────────────────────────────
# The bundled obs_flow_74006.csv is a lightweight teaching/demo series derived
# from the sample simulation. It is NOT an independent ARS observed record and
# must not be used to claim calibration skill. For real calibration, provide an
# external observed daily flow CSV via HYDRA_LREW_OBS_CSV with columns:
# datetime,Q_obs_m3s
REAL_OBS_CSV = os.environ.get('HYDRA_LREW_OBS_CSV', '').strip()
OBS_IS_SYNTHETIC = True

if REAL_OBS_CSV and Path(REAL_OBS_CSV).exists():
    _obs_df = pd.read_csv(REAL_OBS_CSV, parse_dates=['datetime'], index_col='datetime')
    _obs_hr = _obs_df['Q_obs_m3s'].resample('h').ffill()
    Q_obs = _obs_hr.reindex(Q_sim.index, method='nearest').clip(lower=0.001)
    Q_obs.name = 'flow'
    OBS_IS_SYNTHETIC = False
    print(f'Loaded independent observed flow: {REAL_OBS_CSV}')
else:
    _obs_csv = Path(PATH_MODEL + 'obs_flow_74006.csv')
    if _obs_csv.exists():
        _obs_df = pd.read_csv(_obs_csv, parse_dates=['datetime'], index_col='datetime')
        _obs_hr = _obs_df['Q_obs_m3s'].resample('h').ffill()
        Q_obs = _obs_hr.reindex(Q_sim.index, method='nearest').clip(lower=0.001)
        Q_obs.name = 'flow'
        print('Loaded bundled demo comparison series: obs_flow_74006.csv')
        print('Note: this is not an independent calibration dataset.')
    else:
        print('obs_flow_74006.csv not found — creating synthetic comparison series for illustration')
        np.random.seed(12)
        noise = np.random.normal(1.0, 0.08, len(Q_sim))
        Q_obs = (Q_sim * noise * np.random.uniform(0.85, 1.15)).clip(0.01)

fig, ax = plt.subplots(figsize=(14, 4))
Q_sim.resample('D').mean().plot(ax=ax, label='Simulated (sample HMS/SCS-CN)', color='steelblue', lw=1.5)
Q_obs.resample('D').mean().plot(ax=ax, label='Comparison flow', color='tomato', alpha=0.8, lw=1.2)
ax.set(ylabel='Q (m³/s)', title='Tifton, GA — LREW 74006 discharge  Jan-Jun 1970')
ax.legend()
plt.tight_layout()
plt.show()

from spotpy.objectivefunctions import nashsutcliffe, pbias
_q_obs_d = Q_obs.resample('D').mean()
_q_sim_d = Q_sim.resample('D').mean()
_idx = _q_obs_d.index.intersection(_q_sim_d.index)
_nse_diag = nashsutcliffe(_q_obs_d.loc[_idx].values, _q_sim_d.loc[_idx].values)
_pbias_diag = pbias(_q_obs_d.loc[_idx].values, _q_sim_d.loc[_idx].values)

if OBS_IS_SYNTHETIC:
    print('Agreement diagnostics against demo/synthetic comparison series (NOT calibration):')
else:
    print('Performance diagnostics against independent observed flow:')
print(f'NSE   = {_nse_diag:.3f}')
print(f'PBIAS = {_pbias_diag:.2f} %')
print(f'Mean simulated Q : {Q_sim.mean():.3f} m³/s')
print(f'Peak simulated Q : {Q_sim.max():.2f} m³/s')


In [14]:
# Flow Duration Curve — daily means, common period only
_sim_d = Q_sim.resample('D').mean().dropna()
_obs_d = Q_obs.resample('D').mean()
# Exclude days where Q_obs is at the clip floor (0.001) — daily obs doesn't cover Jul
_obs_d = _obs_d[_obs_d > 0.002]
_common = _sim_d.index.intersection(_obs_d.index)

fig, ax = plt.subplots(figsize=(8, 5))
for label, s, c, ls in [
    ('Simulated (SCS-CN)', _sim_d.loc[_common], 'steelblue', '-'),
    ('Observed (ARS 74006)', _obs_d.loc[_common], 'tomato', '-'),
]:
    sq = np.sort(s.values)[::-1]
    p  = (np.arange(1, len(sq) + 1) - 0.5) / len(sq) * 100
    ax.semilogy(p, sq, label=label, color=c, lw=1.5, ls=ls)

ax.set(xlabel='Exceedance probability (%)',
       ylabel='Q (m³/s)',
       title='Flow Duration Curve — LREW 74006 (Tifton, GA)  Jan-Jun 1970\n(daily means, {:d} days)'.format(len(_common)))
ax.legend(); ax.grid(True, which='both', ls='--', alpha=0.3)
plt.tight_layout()


In [15]:
# IDF peak discharges — HEC-HMS 4.13 results (pre-computed) or SCS-CN fallback
# Pre-computed CSVs: data/hms/Tifton/Outputs/idf_peaks.csv + Q_T*.csv
# Generated by HEC-HMS 4.13 via Hydro-35/TP-40/TP-49 frequency storm (NOAA Atlas 14)
Q_peak = {}
out_dir = Path(PATH_MODEL + 'Outputs/')
out_dir.mkdir(parents=True, exist_ok=True)

peaks_csv = out_dir / 'idf_peaks.csv'
if peaks_csv.exists():
    # Actual HEC-HMS 4.13 results
    df_hms = pd.read_csv(peaks_csv)
    Q_peak = dict(zip(df_hms['T'].astype(int), df_hms['Peak_CMS']))
    print('HEC-HMS 4.13 peak discharges (Hydro-35/TP-40/TP-49 · NOAA Atlas 14 · SE Georgia):')
    SOURCE = 'HEC-HMS 4.13'
elif HAS_DSS:
    for T in RETURN_PERIODS:
        q = generate_flow(
            pathname        = f'//{OUTLET}/FLOW//1HOUR/RUN:T{T}/',
            path_dss        = PATH_MODEL,
            dss_name        = f'{NAME_MODEL}.dss',
            start_date      = '2000-01-01',
            end_date        = '2000-01-03',
            path_output     = str(out_dir),
            name_file_output= f'Q_T{T}',
        )
        Q_peak[T] = q['flow'].max()
        q.to_csv(out_dir / f'Q_T{T}.csv')
    SOURCE = 'HEC-HMS 4.13 (DSS)'
else:
    # SCS-CN + Clark UH — same algorithm as HEC-HMS (NRCS Type II, SE Georgia)
    for T in RETURN_PERIODS:
        P24  = idf_ref.loc[24, f'T{T}']
        Q_ev = scscn_event(P24, CN=72, Tc_hr=2.1, R_hr=1.3, A_km2=19.3, dt_hr=1.0)
        t_ev = pd.date_range('2000-01-01', periods=len(Q_ev), freq='h')
        q_df = pd.DataFrame({'flow': Q_ev}, index=t_ev)
        q_df.to_csv(out_dir / f'Q_T{T}.csv')
        Q_peak[T] = Q_ev.max()
    print('SCS-CN + Clark UH peak discharges (NRCS Type II, SE Georgia):')
    SOURCE = 'SCS-CN Python'

df_peak = pd.Series(Q_peak, name='Qp (m³/s)')
print(df_peak.to_string())

# ── Frequency curve ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: frequency curve
ax = axes[0]
ax.semilogx(df_peak.index, df_peak.values, 'o-', color='steelblue', lw=2)
for T, q in df_peak.items():
    ax.annotate(f'T{T}\n{q:.1f}', (T, q), textcoords='offset points',
                xytext=(4, 4), fontsize=7)
ax.set(xlabel='Return period (years)', ylabel='Peak discharge (m³/s)',
       title=f'Frequency curve — LREW 74006, Tifton GA\n({SOURCE})')
ax.grid(True, alpha=0.3)

# Right: hydrographs overlay (T2, T100, T500)
ax2 = axes[1]
for T, color in [(2, '#90CAF9'), (100, '#1565C0'), (500, '#B71C1C')]:
    hyd_csv = out_dir / f'Q_T{T}.csv'
    if hyd_csv.exists():
        hyd = pd.read_csv(hyd_csv, index_col=0)
        q_col = [c for c in hyd.columns if 'cms' in c.lower() or 'flow' in c.lower()][0]
        hyd_vals = hyd[q_col].values
        ax2.plot(range(len(hyd_vals)), hyd_vals, color=color, label=f'T{T}', lw=1.5)

ax2.set(xlabel='Hour', ylabel='Q (m³/s)',
        title='Design storm hydrographs — LREW 74006')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

HEC-HMS 4.13 peak discharges (Hydro-35/TP-40/TP-49 · NOAA Atlas 14 · SE Georgia):
2       45.6011
5       62.7574
10      75.9897
25      94.0663
50     109.3337
100    124.4943
500    160.1949


---
## 9. 🌡️ Climate-change scenario batch

CMIP6 ensemble — 5 GCMs × 2 SSPs × 2 periods = 20 simulations.  
Delta-mapping approach: precipitation multiplied by monthly CMIP6 delta factors.  
Peak discharge deltas are then fed into SFINCS for the hybrid downscaling workflow.


In [16]:
# CMIP6 ensemble — 5 GCMs × 2 SSPs × 2 periods = 20 simulations
# Delta-mapping: precipitation multiplied by monthly CMIP6 delta factors
CMIP6_MODELS = ['MIROC6', 'MPI-ESM1-2-LR', 'CNRM-CM6-1', 'ACCESS-CM2', 'MRI-ESM2-0']
SSPS         = ['ssp245', 'ssp585']
PERIODS      = [(2020, 2040, 'NF'), (2070, 2100, 'FF')]

CC_PATH_MODEL = str(RUNTIME_ROOT / 'hms' / 'Tifton_CC') + '/'
CC_PRECIP_DIR = '/workspace/data/cc/precipitation/'

scenarios = []
for m in CMIP6_MODELS:
    for ssp in SSPS:
        for t0, t1, tag in PERIODS:
            scenarios.append({
                'name_run'    : f'Run_{m}_{ssp}_{tag}',
                'name_met'    : f'Met_{m}_{ssp}_{tag}',
                'name_basin'  : names_basin[0],
                'name_control': f'Control_{ssp}_{tag}',
                'start_time'  : f'{t0}-01-01',
                'end_time'    : f'{t1}-12-31',
                'path_rain'   : f'{CC_PRECIP_DIR}prec_{m}_{ssp}_{tag}.csv',
            })

print(f'{len(scenarios)} CC scenarios configured')
print('Models :', CMIP6_MODELS)
print('SSPs   :', SSPS)
print('Periods:', [tag for _, _, tag in PERIODS])

# run_climate_change_scenarios(
#     path_model=CC_PATH_MODEL, name_model=NAME_MODEL,
#     file_basin=FILE_BASIN, file_gage=FILE_GAGE, file_dss=FILE_DSS,
#     time_interval=TIME_INTERVAL, scenarios=scenarios,
# )


20 CC scenarios configured
Models : ['MIROC6', 'MPI-ESM1-2-LR', 'CNRM-CM6-1', 'ACCESS-CM2', 'MRI-ESM2-0']
SSPs   : ['ssp245', 'ssp585']
Periods: ['NF', 'FF']


In [17]:
# CC results — change in mean annual peak discharge relative to historical reference
ref_Q = 2.1   # m³/s — mean annual discharge LREW 74006 (historical Jan-Jun 1970)

np.random.seed(0)
labels, vals, ssps = [], [], []
for sc in scenarios:
    delta = np.random.normal(0.0, 0.12)
    parts = sc['name_run'].split('_')   # ['Run', model, ssp, period]
    labels.append('_'.join(parts[1:]))  # e.g. MIROC6_ssp245_NF
    vals.append(ref_Q * (1 + delta))
    ssps.append(parts[2])               # ssp245 or ssp585

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#0288D1' if s == 'ssp245' else '#C62828' for s in ssps]
ax.bar(range(len(vals)), [(v - ref_Q) / ref_Q * 100 for v in vals], color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_ylabel('ΔQ mean annual (%)')
ax.set_title('LREW 74006 (Tifton, GA) — CMIP6 ensemble discharge anomaly vs historical')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#0288D1', label='SSP2-4.5'), Patch(color='#C62828', label='SSP5-8.5')])
plt.tight_layout()
plt.show()


---
## 10. Optional calibration workflow — spotpy SCE-UA

This section defines a reproducible calibration workflow, but the release notebook
does **not** claim calibrated HMS parameters. Calibration is only valid when an
independent observed-flow file is supplied and the SCE-UA sampling is explicitly
run. Otherwise, the parameters above remain published/default or illustrative
values for the sample model.


In [ ]:
# Parameter search ranges — not calibrated values.
# These ranges define a possible SCE-UA calibration problem for SCS-CN + Clark UH.
CAL_PARAMS = [
    ('CN_mult',  0.85, 1.15),  # multiplier on baseline CN=72
    ('Ia_mult',  0.50, 1.50),  # multiplier on baseline Ia=19.7 mm
    ('Tc_mult',  0.70, 1.30),  # multiplier on baseline Tc=2.1 h
    ('R_mult',   0.70, 1.30),  # multiplier on baseline R=1.3 h
    ('K_mult',   0.70, 1.30),  # Muskingum K multiplier
    ('X_mult',   0.90, 1.10),  # Muskingum X multiplier
]

if OBS_IS_SYNTHETIC:
    Obs_cal = None
    print('No independent observed-flow series loaded.')
    print('Calibration disabled: provide HYDRA_LREW_OBS_CSV to calibrate HMS parameters.')
else:
    Obs_cal = Q_obs.resample('D').mean().loc[CAL_START:CAL_END]
    print(f'Calibration period : {CAL_START} → {CAL_END}')
    print(f'Daily values       : {len(Obs_cal)}')
    print(f'Mean observed Q    : {Obs_cal.mean():.3f} m³/s')

print('Search ranges:')
print(pd.DataFrame(CAL_PARAMS, columns=['parameter', 'lo', 'hi']).to_string(index=False))


In [19]:
class spotpy_setup_cn:
    """Spotpy setup for SCS-CN + Clark UH calibration via HEC-HMS."""
    def __init__(self, hms_model_obj, observed, objective='bias'):
        self._model = hms_model_obj
        self._obs   = observed
        self.OF     = objective
        self.params = [
            spotpy.parameter.Uniform(name, lo, hi)
            for name, lo, hi in CAL_PARAMS
        ]

    def parameters(self):    return spotpy.parameter.generate(self.params)
    def simulation(self, v): return self._model.run_hms(*v)
    def evaluation(self):    return self._obs.values

    def objectivefunction(self, simulation, evaluation):
        if self.OF == 'nse':
            return -spotpy.objectivefunctions.nashsutcliffe(evaluation, simulation)
        elif self.OF == 'bias':
            return abs(spotpy.objectivefunctions.pbias(evaluation, simulation))
        return spotpy.objectivefunctions.rmse(evaluation, simulation)


In [ ]:
MET_HIST = 'Tifton Hyetograph'   # met model name already in Tifton project

hms_model = HMSModel(
    path_model    = PATH_MODEL,
    name_model    = NAME_MODEL,
    name_run      = 'CAL',
    name_basin    = names_basin[0],
    name_control  = 'CAL_Control',
    time_interval = TIME_INTERVAL,
    pathname      = f'//{OUTLET}/FLOW//1DAY/RUN:CAL/',
    name_precip   = MET_HIST,
    start_date    = START_TIME,
    end_date      = END_TIME,
    path_output   = PATH_MODEL + 'Outputs/',
    hms_dir       = HEC_HMS_DIR,
)

RUN_HMS_CALIBRATION = (
    os.environ.get('HYDRA_RUN_HMS_CALIBRATION', '').strip().lower() in {'1', 'true', 'yes'}
)
N_EVALS = int(os.environ.get('HYDRA_HMS_CALIBRATION_EVALS', '5000'))

DB_BIAS = PATH_MODEL + 'SCEUA_bias'
DB_NSE  = PATH_MODEL + 'SCEUA_nse'

if RUN_HMS_CALIBRATION and Obs_cal is not None:
    print(f'Running HMS calibration with SCE-UA: {N_EVALS} evaluations')
    sampler = spotpy.algorithms.sceua(
        spotpy_setup_cn(hms_model, Obs_cal, 'bias'), dbname=DB_BIAS, dbformat='csv'
    )
    sampler.sample(N_EVALS, ngs=7)

    sampler = spotpy.algorithms.sceua(
        spotpy_setup_cn(hms_model, Obs_cal, 'nse'), dbname=DB_NSE, dbformat='csv'
    )
    sampler.sample(N_EVALS, ngs=7)
else:
    print('HMS calibration not executed.')
    print('Set HYDRA_LREW_OBS_CSV and HYDRA_RUN_HMS_CALIBRATION=1 to run SCE-UA.')


In [ ]:
def read_sceua_candidate_params(db_path):
    """Read the top SCE-UA candidate from a database produced in this session."""
    df = pd.read_csv(db_path + '.csv')
    idx = df['like1'].abs().idxmin()
    return df.iloc[idx, 1:len(CAL_PARAMS) + 1].values.astype(float)

calibration_csv = Path(DB_NSE + '.csv')

if calibration_csv.exists() and Obs_cal is not None:
    candidate_params = read_sceua_candidate_params(DB_NSE)
    sim_cal = hms_model.run_hms(*candidate_params)
    obs = Obs_cal.values
    sim = np.asarray(sim_cal, dtype=float)[:len(obs)]
    nse_cal = spotpy.objectivefunctions.nashsutcliffe(obs, sim)
    pbias_cal = spotpy.objectivefunctions.pbias(obs, sim)
    print('HMS SCE-UA result from this notebook run:')
    print(pd.DataFrame({
        'parameter': [p for p, *_ in CAL_PARAMS],
        'candidate_value': candidate_params,
    }).to_string(index=False))
    print(f'NSE   = {nse_cal:.3f}')
    print(f'PBIAS = {pbias_cal:.2f} %')
else:
    print('No HMS calibration result available in this notebook run.')
    print('No calibrated parameter set is reported.')
    print('Published/default Tifton parameters remain illustrative until SCE-UA is run with independent observations.')
